# Can You Arbitrage Kalshi Against Polymarket US?
If the same outcome trades cheaper on one venue than it sells for on the other, you can buy it on the cheap venue and sell it on the dear one and lock a profit that does not depend on who wins.

## Method
Each matched market is one YES outcome quoted on both venues (a team winning, a player winning), normalised by the collector to a common YES basis. For every moment either order book changes, we line up the latest book on both venues (an as-of join) and ask whether a risk-free trade exists.

Two directions are possible at any instant:
- buy YES on Kalshi at its ask, sell YES on Polymarket at its bid, or
- buy YES on Polymarket at its ask, sell YES on Kalshi at its bid.

The **gross edge** is the better of the two, `max(poly_bid − kalshi_ask, kalshi_bid − poly_ask)`. A positive gross edge means the books cross *across venues* — a locked position for a guaranteed \$1 settlement.

### Costs
Both venues charge a per-contract taker fee that peaks at 50c (see Appendix B):
- Kalshi: `7c × p × (1 − p)`
- Polymarket US: `6c × p × (1 − p)`

An arbitrage takes both legs, so it pays both fees. The **net edge** subtracts them. Only a positive *net* edge is money.

### Freshness
The two books are quoted asynchronously. Each aligned tick carries the age of each leg; the headline analysis requires **both legs be at most 5 seconds old**, so an "edge" is never an artifact of one venue's stale quote. Appendix A shows the effect of this gate.

## Data
Live order-book snapshots from the official Kalshi and Polymarket US websockets, captured by `db/arbitrage/collect_orderbooks.py` and frozen into `db/arbitrage/arb_data.db` by `db/arbitrage/prepare_arb_analysis.py`.

The capture spans **~80 minutes** (2026-07-20 20:40 – 22:00 ET). 50 matched markets were subscribed; **45 saw two-sided activity on both venues**: MLB first-5-innings (F5), plus WTA and ATP tennis. 397,407 raw snapshots collapse to 20,002 top-of-book change events and 19,952 aligned ticks. Activity is overwhelmingly **MLB F5** — live games trading fast — while most tennis matches were nearly idle. Polymarket US did not list full-game MLB moneyline during the window (see Appendix C).

Note on capture quality. Kalshi's high-rate delta feed transiently reconstructs into crossed (impossible) books during fast trading; the collector detects and drops any `bid ≥ ask` state, so no crossed book enters this dataset. That history and fix are documented in `docs/data-collect-specs.md`.

## Setup
### Import Libraries

In [ ]:
%matplotlib inline
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import polars as pl
from IPython.display import Markdown, display

### Connect To DB

In [ ]:
con = duckdb.connect(Path("../db/arbitrage/arb_data.db"), read_only=True)
build = con.sql("SELECT * FROM arb_build_info").pl()
build

### Fees and the Edge
`edges` is the analysis table: every aligned tick with a valid two-sided quote on both venues, its gross and net cross-venue edge (best of the two trade directions), the tradable depth for that direction, and whether both legs are fresh.

In [ ]:
FRESH_S = 5.0   # both legs must be at most this many seconds old

con.execute(f'''
    CREATE OR REPLACE TEMP VIEW edges AS
    WITH v AS (
        SELECT a.*, m.market
        FROM arb_aligned a JOIN arb_matches m USING (match_id)
        WHERE k_bid > 0 AND k_ask < 1 AND p_bid > 0 AND p_ask < 1
    ),
    e AS (
        SELECT *,
            -- direction A: buy YES on Kalshi (ask), sell YES on Poly (bid)
            p_bid - k_ask AS gross_a,
            (p_bid - k_ask)
                - 0.07 * k_ask * (1 - k_ask)
                - 0.06 * p_bid * (1 - p_bid) AS net_a,
            -- direction B: buy YES on Poly (ask), sell YES on Kalshi (bid)
            k_bid - p_ask AS gross_b,
            (k_bid - p_ask)
                - 0.06 * p_ask * (1 - p_ask)
                - 0.07 * k_bid * (1 - k_bid) AS net_b
        FROM v
    )
    SELECT
        match_id, market, ts, k_age_s, p_age_s,
        (k_age_s <= {FRESH_S} AND p_age_s <= {FRESH_S}) AS fresh,
        CAST(greatest(gross_a, gross_b) AS DOUBLE) AS gross,
        CAST(greatest(net_a, net_b) AS DOUBLE)     AS net,
        CAST(CASE WHEN net_a >= net_b THEN least(k_asksz, p_bidsz)
             ELSE least(p_asksz, k_bidsz) END AS DOUBLE) AS depth,
        CASE WHEN net_a >= net_b THEN 'buy Kalshi / sell Poly'
             ELSE 'buy Poly / sell Kalshi' END AS direction
    FROM e
''')
con.sql("SELECT count(*) AS ticks, count(*) FILTER (fresh) AS fresh_ticks FROM edges").pl()

### Chart Style
Colors and matplotlib params, matched to the zh_init site light theme.

In [ ]:
GREEN = "#3d7356"
CLAY = "#c2703d"
INK = "#1c1c1a"
MUTED = "#6e6e6c"
GRID = "#ededeb"
BASELINE = "#d9d9d7"
SURFACE = "#fafaf8"

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "axes.edgecolor": BASELINE, "axes.grid": True,
    "grid.color": GRID, "grid.linewidth": 0.6,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.labelcolor": MUTED,
    "text.color": INK, "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlelocation": "left", "figure.dpi": 110,
})

CENTS = mtick.FuncFormatter(lambda p, _: f"{p * 100:.0f}¢")
PCT = mtick.PercentFormatter(xmax=1, decimals=0)


def fmt_cents(col):
    return pl.format("{}¢", (pl.col(col) * 100).round(2))

def fmt_signed_cents(col):
    v = (pl.col(col) * 100).round(2)
    return (pl.when(v > 0).then(pl.format("+{}¢", v))
              .when(v < 0).then(pl.format("{}¢", v))
              .otherwise(pl.lit("0.0¢")))

def fmt_pct(col):
    return pl.format("{}%", (pl.col(col) * 100).round(1))

def fmt_count(col):
    return pl.col(col).map_elements(lambda n: f"{n:,}", return_dtype=pl.String)

pl.Config.set_tbl_hide_column_data_types(True)
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_tbl_rows(40)

## Cut 1 - Does an edge exist at all?
Every fresh, two-sided tick, summarised. `gross > 0` counts moments the books cross across venues before costs; `net > 0` counts moments a profit survives both taker fees.

In [ ]:
summary = con.sql('''
    SELECT
        count(*) AS fresh_ticks,
        avg((gross > 0)::INT) AS share_gross_pos,
        avg((net > 0)::INT)   AS share_net_pos,
        quantile_cont(gross, 0.5)  AS gross_p50,
        quantile_cont(gross, 0.99) AS gross_p99,
        max(gross) AS gross_max,
        quantile_cont(net, 0.5)  AS net_p50,
        quantile_cont(net, 0.99) AS net_p99,
        max(net) AS net_max
    FROM edges WHERE fresh
''').pl()
summary.select(
    fmt_count("fresh_ticks").alias("fresh ticks"),
    fmt_pct("share_gross_pos").alias("% gross > 0"),
    fmt_pct("share_net_pos").alias("% net > 0"),
    fmt_signed_cents("gross_p50").alias("gross p50"),
    fmt_signed_cents("gross_p99").alias("gross p99"),
    fmt_signed_cents("gross_max").alias("gross max"),
    fmt_signed_cents("net_p50").alias("net p50"),
    fmt_signed_cents("net_p99").alias("net p99"),
    fmt_signed_cents("net_max").alias("net max"),
)

In [ ]:
d = con.sql("SELECT gross, net FROM edges WHERE fresh").pl()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, col, color, title in ((ax1, "gross", CLAY, "Gross edge (before fees)"),
                              (ax2, "net", GREEN, "Net edge (after fees)")):
    x = d[col].to_numpy() * 100
    lo, hi = np.percentile(x, [0.5, 99.5])
    ax.hist(x, bins=np.linspace(lo, hi, 60), color=color, alpha=0.85)
    ax.axvline(0, ls="--", lw=1, color=INK, zorder=3)
    ax.set_title(title, color=INK)
    ax.set_xlabel("edge (cents per contract)")
ax1.set_ylabel("fresh ticks")
fig.suptitle("Distribution of the cross-venue edge", color=INK, x=0.01, ha="left")
fig.tight_layout()
plt.show()

### Opportunities?
Before fees the two venues cross **8.1%** of the time, but the typical gap runs the other way: the median gross edge is **−2.0¢** — it costs about 2¢ to round-trip a contract across venues. After fees only **1.7%** of ticks are positive, the 99th-percentile net edge is **+0.43¢** (indistinguishable from zero), and the single best moment in the hour is **+2.73¢**. Nothing here clears the cost of trading it with room to spare.

## Cut 2 - Which markets?
The capture spans three market types. Do opportunities concentrate in one?

In [ ]:
by_market = con.sql('''
    SELECT market,
        count(DISTINCT match_id) AS matches,
        count(*) AS fresh_ticks,
        avg((net > 0)::INT) AS share_net_pos,
        quantile_cont(net, 0.99) AS net_p99,
        max(net) AS net_max
    FROM edges WHERE fresh
    GROUP BY market ORDER BY share_net_pos DESC
''').pl()
by_market.select(
    pl.col("market"),
    fmt_count("matches").alias("matches"),
    fmt_count("fresh_ticks").alias("fresh ticks"),
    fmt_pct("share_net_pos").alias("% net > 0"),
    fmt_signed_cents("net_p99").alias("net p99"),
    fmt_signed_cents("net_max").alias("net max"),
)

### Opportunities?
MLB F5 carries essentially all the activity and all the marginal edges (8,548 fresh ticks). The tennis matches were nearly idle — 21 fresh WTA ticks and 5 ATP, none positive — so this draft is really a statement about **live MLB F5 pricing**, not tennis.

## Cut 3 - Which matches?
Ranking individual matches by how often a net edge is available isolates whether a few specific books drive everything.

In [ ]:
by_match = con.sql('''
    SELECT match_id, market,
        count(*) AS fresh_ticks,
        avg((net > 0)::INT) AS share_net_pos,
        max(net) AS net_max
    FROM edges WHERE fresh
    GROUP BY match_id, market
    HAVING count(*) >= 50
    ORDER BY share_net_pos DESC LIMIT 20
''').pl()
by_match.select(
    pl.col("match_id"), pl.col("market"),
    fmt_count("fresh_ticks").alias("fresh ticks"),
    fmt_pct("share_net_pos").alias("% net > 0"),
    fmt_signed_cents("net_max").alias("net max"),
)

### Opportunities?
Positive-net moments are scattered across several F5 games (wsh-col 2.6%, cws-tex 1.9%, sf-kc 2.7% of ticks), not concentrated in one mispriced book, and the best per-match net edge never exceeds **2.7¢**. A broad, shallow scatter of tiny edges is what noise around an efficient price looks like — not a standing mispricing one venue is getting wrong.

## Cut 4 - How long does an edge last?
A net edge you cannot reach is not tradable. Grouping consecutive fresh ticks where `net > 0` into windows shows how long each opportunity persists before the books re-converge.

In [ ]:
windows = con.sql('''
    WITH f AS (
        SELECT match_id, ts, (net > 0) AS pos
        FROM edges WHERE fresh
    ),
    grp AS (
        SELECT *,
            row_number() OVER (PARTITION BY match_id ORDER BY ts)
          - row_number() OVER (PARTITION BY match_id, pos ORDER BY ts) AS g
        FROM f
    ),
    win AS (
        SELECT match_id, pos, count(*) AS n_ticks,
               epoch(max(ts) - min(ts)) AS dur_s
        FROM grp GROUP BY match_id, pos, g
    )
    SELECT count(*) AS n_windows,
           quantile_cont(dur_s, 0.5) AS dur_p50,
           quantile_cont(dur_s, 0.9) AS dur_p90,
           max(dur_s) AS dur_max,
           quantile_cont(n_ticks, 0.5) AS ticks_p50
    FROM win WHERE pos
''').pl()
windows

### Opportunities?
The edges do not last. Of **60** positive-net windows, the median lasts a **single tick** (under half a second) and the longest is **7.9 seconds**. Taking both legs across two separate venues inside that window is not realistic, so these moments are not tradable even setting fees aside.

## Cut 5 - How much size?
Even a persistent net edge is only worth the contracts available at the touch. `depth` is the smaller of the two legs' sizes for the profitable direction — the most you could take without walking either book.

In [ ]:
depth = con.sql('''
    SELECT
        quantile_cont(depth, 0.5) AS depth_p50,
        quantile_cont(depth, 0.9) AS depth_p90,
        max(depth) AS depth_max,
        quantile_cont(depth * net, 0.5) AS profit_p50,
        sum(depth * net) AS profit_sum
    FROM edges WHERE fresh AND net > 0
''').pl()
depth

### Opportunities?
Depth is thin too: the median profitable tick offers **0.3 contracts**, the 90th percentile 6.6. Summing net edge × depth across *every* positive-net tick in the hour — capturing all of them, instantly, for free, at full size — totals **$5.14**. That is the ceiling on this hour's cross-venue arbitrage, before a single real-world friction.

## Findings
**No tradable cross-venue arbitrage after fees.** On fresh, two-sided quotes the books cross 8.1% of the time before fees but only 1.7% after, and those net edges are tiny — 99th percentile +0.43¢, max +2.73¢.

**The edges are fleeting and thin.** The median positive-net window is one tick (under 0.5s), the longest 7.9s; median depth is 0.3 contracts. The entire hour's theoretical profit, captured perfectly and cost-free, is **\$5.14**.

**It is MLB F5 or nothing.** Tennis was effectively idle (26 fresh ticks across all matches). Full-game MLB moneyline — the deepest, most liquid market and the one the calibration study used — was not listed on Polymarket US during the window.

**Freshness is the whole ballgame for measurement.** Without requiring both legs young, a stale Polymarket quote fabricates a 33.76¢ 'arbitrage'. Polymarket's book is often seconds to minutes stale (99th-percentile leg age ~21 minutes) while Kalshi's is near-live (~3s). Any cross-venue study here must gate on quote age or it measures latency, not mispricing.

**Read.** Over this hour, Kalshi and Polymarket US priced the same MLB F5 outcomes efficiently relative to each other. The residual cross-venue edge is smaller than the fees to take it and gone in under a second — noise, not opportunity. This is one ~80-minute window on live F5 markets; it is a first look, not the last word.

## Appendix A - Freshness
The as-of alignment forward-fills each venue's last book between updates, so a tick can pair a live book with a stale one. Requiring both legs young removes edges that are really one venue lagging. This compares the edge with and without the 5-second gate.

In [ ]:
fresh_effect = con.sql('''
    SELECT
        'all ticks' AS gate, count(*) AS ticks,
        avg((net > 0)::INT) AS share_net_pos, max(net) AS net_max
    FROM edges
    UNION ALL
    SELECT 'both legs <= 5s', count(*),
        avg((net > 0)::INT), max(net)
    FROM edges WHERE fresh
''').pl()
fresh_effect.select(
    pl.col("gate"), fmt_count("ticks").alias("ticks"),
    fmt_pct("share_net_pos").alias("% net > 0"),
    fmt_signed_cents("net_max").alias("net max"),
)

In [ ]:
ages = con.sql('''
    SELECT quantile_cont(k_age_s, [0.5, 0.9, 0.99]) AS kalshi,
           quantile_cont(p_age_s, [0.5, 0.9, 0.99]) AS poly
    FROM edges
''').pl()
ages

## Appendix B - Fee model
Both venues charge a probability-weighted per-contract taker fee that is largest at 50c and shrinks toward the extremes. An arbitrage pays it on both legs.

In [ ]:
p = np.linspace(0.05, 0.95, 19)
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(p, 7 * p * (1 - p), "o-", ms=3, color=GREEN, label="Kalshi (7c)")
ax.plot(p, 6 * p * (1 - p), "o-", ms=3, color=CLAY, label="Polymarket US (6c)")
ax.set_xlabel("contract price")
ax.set_ylabel("taker fee (cents / contract)")
ax.set_title("Per-contract taker fee by price", color=INK)
ax.xaxis.set_major_formatter(CENTS)
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

## Appendix C - Coverage and limitations
- **Markets.** Polymarket US did not list full-game MLB moneyline during the capture window, so this draft covers MLB first-5-innings (F5) and WTA/ATP tennis moneylines — whatever was live on both venues.
- **Window.** A single ~1-hour capture. Not a claim about other days or markets.
- **Direction.** Every match here resolves `kalshi_yes_eq_poly_yes`; the YES side was fixed from Polymarket's `marketSides` (`long: true`) and the Kalshi sub-market title.
- **Executability.** The edge is computed at the touch and assumes taker fills on both legs at the quoted top-of-book for the available depth; it ignores latency, partial fills, and that taking one leg can move the other.
- **Capture quality.** Crossed (impossible) reconstructed Kalshi books are dropped at capture, so a dropped state leaves a short gap rather than a bad print.

In [ ]:
con.close()